In [0]:
from pyspark.sql import functions as F
import builtins

DATABASE_NAME = "smart_data_deduplication"

# 1. Total incoming records
total_records = spark.table(
    f"{DATABASE_NAME}.staging_records"
).count()

# 2. Validation counts
passed_validation = (
    valid_df.count()
    if "valid_df" in globals()
    else spark.table(
        f"{DATABASE_NAME}.validated_records"
    ).count()
)
failed_validation = total_records - passed_validation

# 3. Count repaired records
# Set repaired_df in your repair notebook.
repaired_records = (
    repaired_df.count()
    if "repaired_df" in globals()
    else 0
)

# 4. Count quarantined records
# Set quarantine_df in your quarantine notebook.
quarantined_records = (
    quarantine_df.count()
    if "quarantine_df" in globals()
    else 0
)

# 5. Successfully processed
successfully_processed = passed_validation + repaired_records

# Avoid counting quarantined records as successful
successfully_processed = max(
    0, successfully_processed - quarantined_records
)

# 6. Percentage calculation
def percentage(value):
    if total_records == 0:
        return 0.0
    return builtins.round(
        (value / total_records) * 100, 2
    )

# 7. Create Table 6
table6_data = [
    ("Passed validation", passed_validation,
     percentage(passed_validation)),

    ("Failed validation", failed_validation,
     percentage(failed_validation)),

    ("Repaired records", repaired_records,
     percentage(repaired_records)),

    ("Quarantined records", quarantined_records,
     percentage(quarantined_records)),

    ("Successfully processed", successfully_processed,
     percentage(successfully_processed))
]

table6_df = spark.createDataFrame(
    table6_data,
    ["Processing Category", "Number of Records", "Percentage"]
)

table6_df = table6_df.withColumn(
    "Percentage",
    F.concat(
        F.format_number(F.col("Percentage"), 2),
        F.lit("%")
    )
)

print("TABLE 6: VALIDATION AND QUARANTINE RESULTS")
print("Total incoming records:", total_records)

display(table6_df)

# 8. Save results (column names cannot contain spaces)
table6_df.toDF(
    "processing_category",
    "number_of_records",
    "percentage"
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{DATABASE_NAME}.validation_quarantine_summary"
    )

print("Table 6 saved successfully!")

TABLE 6: VALIDATION AND QUARANTINE RESULTS
Total incoming records: 60


Processing Category,Number of Records,Percentage
Passed validation,54,90.00%
Failed validation,6,10.00%
Repaired records,0,0.00%
Quarantined records,0,0.00%
Successfully processed,54,90.00%


Table 6 saved successfully!
